# GPU-Fuzzy Trading Pipeline — Run

Local Jupyter runner for the full pipeline (same as `python -m gpu_fuzzy_trader.run_pipeline`).

**Data (only these files):**
- `data/train_new.csv` — Phases 1–4 · OHLCV + `ff_*` features
- `data/test_new.csv` — Phase 5 OOS · matching OHLCV + `ff_*` features


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

def _find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "gpu_fuzzy_trader").is_dir() and (
            candidate / "data" / "train_new.csv"
        ).is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find repo root (gpu_fuzzy_trader/ + data/train_new.csv). "
        "Open the notebook from the trading_platform directory."
    )

ROOT = _find_repo_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("cwd:", Path.cwd())
print("python:", sys.executable)

from gpu_fuzzy_trader.run_pipeline import Pipeline_Orchestrator, _print_run_summary

print("Pipeline_Orchestrator import OK")


cwd: /home/danaee/trading_platform
python: /home/danaee/trading_platform/.venv/bin/python
Pipeline_Orchestrator import OK


In [2]:
import pandas as pd
from gpu_fuzzy_trader import config as cfg

train_path = Path(cfg.TRAIN_CSV_PATH)
test_path = Path(cfg.TEST_CSV_PATH)

assert train_path.is_file(), f"Missing {train_path} (expected data/train_new.csv)"
assert test_path.is_file(), f"Missing {test_path} (expected data/test_new.csv)"


def _datetime_span(path: Path) -> tuple[pd.Timestamp, pd.Timestamp, int]:
    """Datetime span without loading feature columns (files are chronological)."""
    # usecols keeps memory small vs full CSV (~700k timestamps ≈ a few MB)
    dt = pd.to_datetime(pd.read_csv(path, usecols=["datetime"])["datetime"])
    return dt.iloc[0], dt.iloc[-1], len(dt)


for label, path in (("train", train_path), ("test", test_path)):
    t0, t1, n = _datetime_span(path)
    print(f"{label}: {path}  rows={n:,}  {t0} → {t1}")

print("Expected: train 2024-01-01→2024-08-31, test 2024-09-01→2025-01-31")


train: data/train_new.csv  rows=140,352  2024-01-01 00:00:00 → 2025-12-31 23:45:00
test: data/test_new.csv  rows=39,152  2026-01-01 00:00:00 → 2026-07-23 21:45:00
Expected: train 2024-01-01→2024-08-31, test 2024-09-01→2025-01-31


In [3]:
# Same defaults as CLI: full rerun into outputs/
OUTPUT_DIR = "outputs"
FORCE = True  # True = rerun phases 1–4; False = skip when valid artifacts exist (--resume)

print(f"OUTPUT_DIR={OUTPUT_DIR!r}  FORCE={FORCE}")


OUTPUT_DIR='outputs'  FORCE=True


In [ ]:
orchestrator = Pipeline_Orchestrator(output_dir=OUTPUT_DIR)
results = orchestrator.run(force=FORCE)
_print_run_summary(results, phase=None, log_path=orchestrator._log_path)
print(f"\nStructured log: {orchestrator._log_path}")
results


## Outputs & evaluation

After a successful run, expect:

| Path | Phase |
| --- | --- |
| `outputs/selected_features_{long,short}.json` | 1 |
| `pools/phase2_{long,short}_pool.json` | 2 |
| `outputs/long.json`, `outputs/short.json` | 3–4 / RB Governor |
| `outputs/reports/test_*` | 5 |
| `outputs/pipeline.log` | all |

Evaluate strategies with **`evaluator_v5.ipynb`** (uses `data/train_new.csv` schema + `data/test_new.csv`).
